# 🛸 Complete Unsupervised Anomaly Detection: All Models Benchmark (Epochs=15)

Evaluates **all 12 models** (8 Pointwise Classical + 4 Deep Sequence Autoencoders) across the complete feature progression:
1. **Raw Coordinates (6)**: `latitude`, `longitude`, `course`, `ground_speed`, `vertical_speed`, `height`
2. **Raw No Coordinates (4)**: `course`, `ground_speed`, `vertical_speed`, `height`
3. **Raw Engineered (8)**: `course`, `ground_speed`, `vertical_speed`, `height`, `acceleration`, `vertical_acceleration`, `turn_rate`, `path_curvature`
4. **Pure Kinematics Baseline (8)**: `height`, `ground_speed`, `vertical_speed`, `acceleration`, `turn_rate`, `path_curvature`, `heading_speed_consistency`, `motion_smoothness`
5. **Standard Baseline (10)**: Kinematics 8 + `prediction_error`, `yaw_acceleration`
6. **Noise Texture (13)**: Baseline 10 + `prediction_error_autocorrelation`, `position_residual_std`, `speed_spectral_entropy`
7. **Baseline + Cross-Correlation (13)**: Baseline 10 + `corr_speed_turn`, `corr_accel_turn`, `corr_vert_speed`
8. **Full Cross-Correlation & Noise Texture (16)**: Noise Texture 13 + 3 Cross-Correlations

### Models Evaluated (12 Total):
- **Pointwise Classical (8)**: `Isolation Forest`, `GMM`, `Mahalanobis`, `One-Class SVM`, `PCA`, `K-Means`, `DBSCAN`, `KNN`
- **Deep Sequence Autoencoders (4)**: `Dense AE`, `GRU AE`, `TCN AE`, `CNN-GRU AE`

### Datasets Evaluated (7 Classes):
- `Normal DJI` (Clean baseline TNR), `Real ESP32`, `Sim Baseline`, `Sim Easy`, `Sim Medium`, `Sim Hard`, `Sim Geometry`.

## 1. Setup Working Directory & Environment

In [ ]:
from pathlib import Path
import os
import sys
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from presets.run_unsupervised_pipeline import FEATURE_SETS, run_experiment
from implement.utils.helper import get_output_dir

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Operating Directory: {os.getcwd()}")
print(f"Compute Device: {device} | PyTorch: {torch.__version__}")
print(f"Available Feature Presets: {list(FEATURE_SETS.keys())}")

## 2. Experiment 1: Raw Coordinates (6 Features — All Models)

In [ ]:
df_exp1_rc = run_experiment(
    exp_name="all_models_raw_coords",
    feature_list=FEATURE_SETS["raw_coords"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 3. Experiment 2: Raw No Coordinates (4 Features — All Models)

In [ ]:
df_exp2_rnc = run_experiment(
    exp_name="all_models_raw_no_coords",
    feature_list=FEATURE_SETS["raw_no_coords"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 4. Experiment 3: Raw Engineered (8 Features — All Models)

In [ ]:
df_exp3_re = run_experiment(
    exp_name="all_models_raw_engineered",
    feature_list=FEATURE_SETS["raw_engineered"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 5. Experiment 4: Pure Kinematics Baseline (8 Features — All Models)

In [ ]:
df_exp4_b8 = run_experiment(
    exp_name="all_models_baseline_8",
    feature_list=FEATURE_SETS["baseline_8"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 6. Experiment 5: Standard Baseline (10 Features — All Models)

In [ ]:
df_exp5_b10 = run_experiment(
    exp_name="all_models_baseline_10",
    feature_list=FEATURE_SETS["baseline_10"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 7. Experiment 6: Noise Texture Extended (13 Features — All Models)

In [ ]:
df_exp6_nt13 = run_experiment(
    exp_name="all_models_noise_texture_13",
    feature_list=FEATURE_SETS["noise_texture_13"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 8. Experiment 7: Baseline + Cross-Correlation (13 Features — All Models)

In [ ]:
df_exp7_bc13 = run_experiment(
    exp_name="all_models_baseline_corr_13",
    feature_list=FEATURE_SETS["baseline_corr_13"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 9. Experiment 8: Full Cross-Correlation & Noise Texture (16 Features — All Models)

In [ ]:
df_exp8_cc16 = run_experiment(
    exp_name="all_models_correlation_16",
    feature_list=FEATURE_SETS["correlation_16"],
    model_family="all",
    epochs=15,
    batch_size=64,
    patience=7,
    window_len=20,
    fresh_cache=True
)

## 10. Comprehensive Multi-Experiment Summary & Aggregate Comparison

In [ ]:
experiments = [
    ("Raw Coords (6)", "all_models_raw_coords"),
    ("Raw No Coords (4)", "all_models_raw_no_coords"),
    ("Raw Engineered (8)", "all_models_raw_engineered"),
    ("Baseline Kinematics (8)", "all_models_baseline_8"),
    ("Baseline Standard (10)", "all_models_baseline_10"),
    ("Noise Texture (13)", "all_models_noise_texture_13"),
    ("Baseline + Corr (13)", "all_models_baseline_corr_13"),
    ("Cross-Correlation (16)", "all_models_correlation_16")
]

agg_frames = []
for label, exp in experiments:
    p = get_output_dir() / "pipeline_experiments" / exp / f"{exp}_aggregate.csv"
    if p.exists():
        df = pd.read_csv(p)
        df.insert(0, "Feature Preset", label)
        agg_frames.append(df)

if agg_frames:
    full_agg_df = pd.concat(agg_frames, ignore_index=True)
    summary_save_path = get_output_dir() / "all_models_feature_progression_aggregate_summary.csv"
    full_agg_df.to_csv(summary_save_path, index=False)
    print(f"✅ Aggregated summary across all feature progressions saved to: {summary_save_path}")
    display(full_agg_df)
else:
    print("Run the experiment cells above to produce aggregate summaries.")